In [10]:
from dotenv import load_dotenv
import os
import clickhouse_connect
import pandas as pd
from datetime import datetime


load_dotenv()

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST'),
    port=int(os.getenv('CLICKHOUSE_PORT', 8123)), 
    username=os.getenv('CLICKHOUSE_USERNAME'),
    password=os.getenv('CLICKHOUSE_PASSWORD'),
    database=os.getenv('CLICKHOUSE_DATABASE')
)

In [11]:
# shipment_dataset_query = """ 
# SELECT 
#     bni.quantity_in,

#     -- From bills
#     b.bill_date,
#     b.due_date,
#     b.bill_status,
#     b.eta,
#     b.bill_type,
#     b.seal,
#     b.fda_status,
#     b.coo,
#     b.customs_broker,
#     b.receipt_date,
#     b.inventory,
#     b.gross_weight_lb,
#     b.ocean_freight,
#     b.transshipment_date_ett,
#     b.tariff_amount,
#     b.tariff_type,
#     b.place_of_delivery,
#     b.shipping_port,
#     b.consignee,

#     -- From items
#     i.purchase_price,
#     i.sales_price,
#     i.opening_stock,
#     i.cases AS item_cases,
#     i.product_type,
#     i.product_category,
#     i.brand,
#     i.manufacturer,
#     i.sales_account,
#     i.size AS item_size,
#     i.item_type,
#     i.cooking_style,
#     i.soaking_style,
#     i.yield_percentage,
#     i.status

# FROM 
#     zoho_books_analytics.batch_number_in bni
# JOIN 
#     zoho_books_analytics.bills b 
#     ON bni.bill_id = b.bill_id

# JOIN 
#     zoho_books_analytics.items i 
#     ON bni.product_id = i.item_id

# ORDER BY 
#     bni.created_time DESC

# """
shipment_dataset_query = """ 
SELECT 
    bni.quantity_in,

    -- From bills
    b.bill_date,
    b.due_date,
    b.bill_status,
    b.eta AS eta,
    b.bill_type,
    b.seal,
    b.fda_status,
    b.coo,
    b.customs_broker,
    b.receipt_date,
    b.inventory,
    b.gross_weight_lb,
    b.ocean_freight,
    b.transshipment_date_ett,
    b.tariff_amount,
    b.tariff_type AS tariff_type,
    b.place_of_delivery,
    b.shipping_port,
    b.consignee,

    -- From items
    i.purchase_price,
    i.sales_price,
    i.opening_stock,
    i.cases AS item_cases,
    i.product_type,
    i.product_category,
    i.brand,
    i.manufacturer,
    i.sales_account,
    i.size AS item_size,
    i.item_type,
    i.cooking_style,
    i.soaking_style,
    i.yield_percentage,
    i.status

FROM 
    zoho_books_analytics.batch_number_in bni
JOIN 
    zoho_books_analytics.bills b 
    ON bni.bill_id = b.bill_id
JOIN 
    zoho_books_analytics.bill_item bi 
    ON bi.bill_id = b.bill_id
JOIN 
    zoho_books_analytics.purchase_orders po  
    ON po.purchase_order_number = b.purchase_order
JOIN 
    zoho_books_analytics.items i 
    ON i.item_id = bi.product_id
JOIN 
    zoho_books_analytics.customer_item_mapping cim 
    ON i.sku = cim.az_sku

WHERE 
    cim.customer_name LIKE 'Walmart%'
    AND po.po_commited != 'Direct Sale'

ORDER BY 
    bni.created_time DESC
"""


In [12]:
result = client.query(shipment_dataset_query)

shipment_dataset_df = pd.DataFrame(result.result_rows, columns=[col for col in result.column_names])

In [13]:
shipment_dataset_df.columns

Index(['quantity_in', 'bill_date', 'due_date', 'bill_status', 'eta',
       'bill_type', 'seal', 'fda_status', 'coo', 'customs_broker',
       'receipt_date', 'inventory', 'gross_weight_lb', 'ocean_freight',
       'transshipment_date_ett', 'tariff_amount', 'tariff_type',
       'place_of_delivery', 'shipping_port', 'consignee', 'purchase_price',
       'sales_price', 'opening_stock', 'item_cases', 'product_type',
       'product_category', 'brand', 'manufacturer', 'sales_account',
       'item_size', 'item_type', 'cooking_style', 'soaking_style',
       'yield_percentage', 'status'],
      dtype='object')

In [14]:
shipment_dataset_df['eta'] = pd.to_datetime(shipment_dataset_df['eta'], format='%d %b %Y', errors='raise')
shipment_dataset_df['receipt_date'] = pd.to_datetime(shipment_dataset_df['receipt_date'], format='%d %b %Y', errors='raise')
shipment_dataset_df['due_date'] = pd.to_datetime(shipment_dataset_df['due_date'], format='%d %b %Y', errors='raise')
shipment_dataset_df['bill_date'] = pd.to_datetime(shipment_dataset_df['bill_date'], format='%d %b %Y', errors='raise')
shipment_dataset_df['transshipment_date_ett'] = pd.to_datetime(shipment_dataset_df['transshipment_date_ett'], format='%d %b %Y', errors='raise')


# # Replace NaT (null) with today's date
# shipment_dataset_df['receipt_date'].fillna(pd.Timestamp(datetime.today().strftime('%Y-%m-%d')), inplace=True)

shipment_dataset_df.dropna(subset='receipt_date', inplace=True)

# Compute shipment delay in days
shipment_dataset_df['shipment_delay_days'] = (
    shipment_dataset_df['receipt_date'] - shipment_dataset_df['eta']
).dt.days

In [15]:
threshold = 5

In [16]:
shipment_dataset_df['shipment_classified'] = shipment_dataset_df['shipment_delay_days'].apply(
    lambda x: 'on_time' if x <= threshold else 'delayed')

In [17]:
# shipment_dataset_df.drop_duplicates(inplace=True)

In [18]:
shipment_dataset_df.to_csv('./results/raw_shipment_classification_dataset.csv', index=False)